In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
from collections import defaultdict
import random
from sklearn.metrics import accuracy_score

In [2]:
def prepare_sequence(seq, to_ix):
    idxs = [to_ix[w] for w in seq]
    return torch.tensor(idxs, dtype=torch.long)


import nltk
nltk.download('brown')
nltk.download('universal_tagset')

def load_brown_corpus(num_sentences=1500):
    """Load Brown corpus with universal tagset"""
    from nltk.corpus import brown

    tagged_sents = brown.tagged_sents(tagset='universal')

    training_data = []
    for tagged_sent in tagged_sents[:num_sentences]:
        words = [word for word, tag in tagged_sent]
        tags = [tag for word, tag in tagged_sent]
        training_data.append((words, tags))

    return training_data

print("Loading Brown corpus...")
training_data = load_brown_corpus(num_sentences=1500)
print(f"Loaded {len(training_data)} sentences")

word_to_ix = {}
# For each words-list (sentence) and tags-list in each tuple of training_data
for sent, tags in training_data:
    for word in sent:
        if word not in word_to_ix:  # word has not been assigned an index yet
            word_to_ix[word] = len(word_to_ix)  # Assign each word with a unique index
print(word_to_ix)
tag_to_ix = {"DET": 0, "NN": 1, "V": 2}  # Assign each tag with a unique index

# These will usually be more like 32 or 64 dimensional.
# We will keep them small, so we can see how the weights change as we train.
EMBEDDING_DIM = 6
HIDDEN_DIM = 6

[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Unzipping corpora/brown.zip.
[nltk_data] Downloading package universal_tagset to /root/nltk_data...
[nltk_data]   Unzipping taggers/universal_tagset.zip.


Loading Brown corpus...
Loaded 1500 sentences
{'The': 0, 'Fulton': 1, 'County': 2, 'Grand': 3, 'Jury': 4, 'said': 5, 'Friday': 6, 'an': 7, 'investigation': 8, 'of': 9, "Atlanta's": 10, 'recent': 11, 'primary': 12, 'election': 13, 'produced': 14, '``': 15, 'no': 16, 'evidence': 17, "''": 18, 'that': 19, 'any': 20, 'irregularities': 21, 'took': 22, 'place': 23, '.': 24, 'jury': 25, 'further': 26, 'in': 27, 'term-end': 28, 'presentments': 29, 'the': 30, 'City': 31, 'Executive': 32, 'Committee': 33, ',': 34, 'which': 35, 'had': 36, 'over-all': 37, 'charge': 38, 'deserves': 39, 'praise': 40, 'and': 41, 'thanks': 42, 'Atlanta': 43, 'for': 44, 'manner': 45, 'was': 46, 'conducted': 47, 'September-October': 48, 'term': 49, 'been': 50, 'charged': 51, 'by': 52, 'Superior': 53, 'Court': 54, 'Judge': 55, 'Durwood': 56, 'Pye': 57, 'to': 58, 'investigate': 59, 'reports': 60, 'possible': 61, 'hard-fought': 62, 'won': 63, 'Mayor-nominate': 64, 'Ivan': 65, 'Allen': 66, 'Jr.': 67, 'Only': 68, 'a': 69, 'r

In [3]:
random.shuffle(training_data)
train_size = int(0.9 * len(training_data))  # Use 90% for training
train_data = training_data[:train_size]
test_data = training_data[train_size:]

In [4]:
def build_vocab(data):
    """Build vocabulary with UNK token for unseen words"""
    word_to_ix = {"<UNK>": 0}  # Unknown word token
    tag_to_ix = {}

    for sent, tags in data:
        for word in sent:
            if word not in word_to_ix:
                word_to_ix[word] = len(word_to_ix)
        for tag in tags:
            if tag not in tag_to_ix:
                tag_to_ix[tag] = len(tag_to_ix)

    return word_to_ix, tag_to_ix

In [5]:
word_to_ix, tag_to_ix = build_vocab(training_data)
ix_to_tag = {v: k for k, v in tag_to_ix.items()}

In [6]:
EMBEDDING_DIM = 50
HIDDEN_DIM = 64
DROPOUT_RATE = 0.2
LEARNING_RATE = 0.01

In [7]:
def prepare_sequence(seq, to_ix, unk_token="<UNK>"):
    """Converts sequence to tensor with UNK handling"""
    idxs = [to_ix.get(w, to_ix[unk_token]) for w in seq]
    return torch.tensor(idxs, dtype=torch.long)

def calculate_accuracy(predictions, targets):
    """Calculate accuracy between predictions and targets"""
    pred_tags = [torch.argmax(pred).item() for pred in predictions]
    target_tags = targets.tolist()
    return accuracy_score(target_tags, pred_tags)

In [8]:
class LSTMTagger(nn.Module):

     def __init__(self, embedding_dim, hidden_dim, vocab_size, tagset_size, dropout_rate=0.2):
        super(LSTMTagger, self).__init__()
        self.hidden_dim = hidden_dim

        # Word embeddings with dropout
        self.word_embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.embedding_dropout = nn.Dropout(dropout_rate)

        # Bidirectional LSTM for better context
        self.lstm = nn.LSTM(embedding_dim, hidden_dim,
                           bidirectional=True, batch_first=False, dropout=dropout_rate)

        # Linear layer (bidirectional → double hidden_dim)
        self.hidden2tag = nn.Linear(hidden_dim * 2, tagset_size)
        self.dropout = nn.Dropout(dropout_rate)

     def forward(self, sentence):
        # Embedding lookup with dropout
        embeds = self.word_embeddings(sentence)
        embeds = self.embedding_dropout(embeds)

        # LSTM forward pass (seq_len, batch, features)
        lstm_out, _ = self.lstm(embeds.view(len(sentence), 1, -1))

        # Apply dropout
        lstm_out = self.dropout(lstm_out)

        # Project to tag space
        tag_space = self.hidden2tag(lstm_out.view(len(sentence), -1))

        # Log softmax for NLL loss
        tag_scores = F.log_softmax(tag_space, dim=1)
        return tag_scores


In [9]:
model = LSTMTagger(EMBEDDING_DIM, HIDDEN_DIM,
                          len(word_to_ix), len(tag_to_ix), DROPOUT_RATE)

print(f"Model parameters: {sum(p.numel() for p in model.parameters())}")
print(f"Vocabulary size: {len(word_to_ix)}")
print(f"Tag set size: {len(tag_to_ix)}")
print(f"Tags: {list(tag_to_ix.keys())}")

Model parameters: 378640
Vocabulary size: 6354
Tag set size: 12
Tags: ['NOUN', '.', 'NUM', 'ADP', 'CONJ', 'PRON', 'VERB', 'ADV', 'ADJ', 'DET', 'PRT', 'X']


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn(


In [10]:
loss_function = nn.NLLLoss()  # Negative Log-Likelihood
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)


In [11]:
with torch.no_grad():
    inputs = prepare_sequence(training_data[0][0], word_to_ix)
    tag_scores = model(inputs)
    print("Before training:\n", tag_scores)

Before training:
 tensor([[-2.5378, -2.4362, -2.5150, -2.4387, -2.5087, -2.4434, -2.5493, -2.5744,
         -2.3362, -2.5281, -2.4441, -2.5326],
        [-2.6323, -2.4648, -2.5672, -2.5319, -2.3499, -2.4003, -2.5064, -2.5155,
         -2.4372, -2.5366, -2.3629, -2.5555],
        [-2.5322, -2.5216, -2.3851, -2.6193, -2.3725, -2.4722, -2.6198, -2.5713,
         -2.3058, -2.4691, -2.3857, -2.6314],
        [-2.4706, -2.5189, -2.4104, -2.5643, -2.4320, -2.5492, -2.4999, -2.5870,
         -2.4037, -2.4233, -2.4060, -2.5818]])


In [12]:
def train_model(model, train_data, test_data, epochs=300):
    loss_function = nn.NLLLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=100, gamma=0.8)

    train_losses = []
    test_accuracies = []

    print("Starting training...")

    for epoch in range(epochs):
        # Training phase
        model.train()
        total_loss = 0

        for sentence, tags in train_data:
            model.zero_grad()

            # Prepare data
            sentence_in = prepare_sequence(sentence, word_to_ix)
            targets = torch.tensor([tag_to_ix[tag] for tag in tags], dtype=torch.long)
            # Forward pass
            tag_scores = model(sentence_in)
            loss = loss_function(tag_scores, targets)

            # Backward pass
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        scheduler.step()
        avg_loss = total_loss / len(train_data)
        train_losses.append(avg_loss)

        # Validation phase
        if epoch % 50 == 0 or epoch == epochs - 1:
            model.eval()
            test_acc = evaluate_model(model, test_data)
            test_accuracies.append(test_acc)
            print(f"Epoch {epoch:3d}: Loss = {avg_loss:.4f}, Test Accuracy = {test_acc:.3f}")

    return train_losses, test_accuracies

In [13]:
def evaluate_model(model, test_data):
    """Evaluate model on test data"""
    model.eval()
    all_predictions = []
    all_targets = []

    with torch.no_grad():
        for sentence, tags in test_data:
            sentence_in = prepare_sequence(sentence, word_to_ix)
            targets = torch.tensor([tag_to_ix[tag] for tag in tags], dtype=torch.long)
            tag_scores = model(sentence_in)
            predictions = [torch.argmax(score).item() for score in tag_scores]

            all_predictions.extend(predictions)
            all_targets.extend(targets.tolist())

    return accuracy_score(all_targets, all_predictions)

In [14]:
model = LSTMTagger(EMBEDDING_DIM, HIDDEN_DIM,
                          len(word_to_ix), len(tag_to_ix), DROPOUT_RATE)


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn(


In [15]:
print(f"Model parameters: {sum(p.numel() for p in model.parameters())}")
print(f"Vocabulary size: {len(word_to_ix)}")
print(f"Tag set size: {len(tag_to_ix)}")
print(f"Tags: {list(tag_to_ix.keys())}")

Model parameters: 378640
Vocabulary size: 6354
Tag set size: 12
Tags: ['NOUN', '.', 'NUM', 'ADP', 'CONJ', 'PRON', 'VERB', 'ADV', 'ADJ', 'DET', 'PRT', 'X']


In [16]:
train_losses, test_accuracies = train_model(model, train_data, test_data, epochs=500)

Starting training...
Epoch   0: Loss = 0.7491, Test Accuracy = 0.869
Epoch  50: Loss = 0.0597, Test Accuracy = 0.919
Epoch 100: Loss = 0.0446, Test Accuracy = 0.926
Epoch 150: Loss = 0.0354, Test Accuracy = 0.923
Epoch 200: Loss = 0.0286, Test Accuracy = 0.927
Epoch 250: Loss = 0.0198, Test Accuracy = 0.920
Epoch 300: Loss = 0.0207, Test Accuracy = 0.925
Epoch 350: Loss = 0.0147, Test Accuracy = 0.925
Epoch 400: Loss = 0.0127, Test Accuracy = 0.926
Epoch 450: Loss = 0.0105, Test Accuracy = 0.927
Epoch 499: Loss = 0.0091, Test Accuracy = 0.926


In [17]:
def predict_sentence(model, sentence, word_to_ix, ix_to_tag):
    """Predict POS tags for a given sentence"""
    model.eval()
    with torch.no_grad():
        # Handle words not in vocabulary
        processed_sentence = []
        for word in sentence:
            if word in word_to_ix:
                processed_sentence.append(word)
            else:
                processed_sentence.append("<UNK>")
                print(f"Warning: '{word}' not in vocabulary, using <UNK>")

        sentence_in = prepare_sequence(processed_sentence, word_to_ix)
        tag_scores = model(sentence_in)

        predictions = []
        confidences = []
        for score in tag_scores:
            pred_idx = torch.argmax(score).item()
            confidence = torch.softmax(score, dim=0)[pred_idx].item()
            predictions.append(ix_to_tag[pred_idx])
            confidences.append(confidence)

        return predictions, confidences

# Test on training data
print("\n" + "="*50)
print("TESTING ON TRAINING DATA")
print("="*50)

for i, (sentence, true_tags) in enumerate(train_data):
    predicted_tags, confidences = predict_sentence(model, sentence, word_to_ix, ix_to_tag)

    print(f"\nSentence {i+1}: {' '.join(sentence)}")
    print("Word       | True Tag | Pred Tag | Confidence")
    print("-" * 45)

    for word, true_tag, pred_tag, conf in zip(sentence, true_tags, predicted_tags, confidences):
        status = "✓" if true_tag == pred_tag else "✗"
        print(f"{word:10} | {true_tag:8} | {pred_tag:8} | {conf:.3f} {status}")


Le flux de sortie a été tronqué et ne contient que les 5000 dernières lignes.
police     | NOUN     | NOUN     | 1.000 ✓
burglary   | NOUN     | NOUN     | 1.000 ✓
trial      | NOUN     | NOUN     | 1.000 ✓
made       | VERB     | VERB     | 1.000 ✓
statements | NOUN     | NOUN     | 1.000 ✓
indicating | VERB     | VERB     | 1.000 ✓
their      | DET      | DET      | 1.000 ✓
guilt      | NOUN     | NOUN     | 1.000 ✓
at         | ADP      | ADP      | 1.000 ✓
the        | DET      | DET      | 1.000 ✓
time       | NOUN     | NOUN     | 1.000 ✓
of         | ADP      | ADP      | 1.000 ✓
their      | DET      | DET      | 1.000 ✓
arrest     | NOUN     | NOUN     | 1.000 ✓
,          | .        | .        | 1.000 ✓
Judge      | NOUN     | NOUN     | 1.000 ✓
James      | NOUN     | NOUN     | 1.000 ✓
B.         | NOUN     | NOUN     | 1.000 ✓
Parsons    | NOUN     | NOUN     | 1.000 ✓
was        | VERB     | VERB     | 1.000 ✓
told       | VERB     | VERB     | 1.000 ✓
in         | ADP   

In [18]:
print("\n" + "="*50)
print("TESTING ON NEW SENTENCES")
print("="*50)

new_sentences = [
    "The quick brown fox jumps".split(),
    "She reads books every day".split(),
    "Dogs bark at strangers".split()
]

for sentence in new_sentences:
    predicted_tags, confidences = predict_sentence(model, sentence, word_to_ix, ix_to_tag)

    print(f"\nSentence: {' '.join(sentence)}")
    print("Word       | Pred Tag | Confidence")
    print("-" * 32)

    for word, pred_tag, conf in zip(sentence, predicted_tags, confidences):
        print(f"{word:10} | {pred_tag:8} | {conf:.3f}")

print(f"\nFinal Test Accuracy: {test_accuracies[-1]:.3f}")


TESTING ON NEW SENTENCES

Sentence: The quick brown fox jumps
Word       | Pred Tag | Confidence
--------------------------------
The        | DET      | 1.000
quick      | ADJ      | 1.000
brown      | NOUN     | 0.993
fox        | NOUN     | 0.981
jumps      | NOUN     | 0.921

Sentence: She reads books every day
Word       | Pred Tag | Confidence
--------------------------------
She        | PRON     | 1.000
reads      | VERB     | 1.000
books      | NOUN     | 0.998
every      | DET      | 1.000
day        | NOUN     | 1.000

Sentence: Dogs bark at strangers
Word       | Pred Tag | Confidence
--------------------------------
Dogs       | ADJ      | 0.954
bark       | NOUN     | 0.726
at         | ADP      | 1.000
strangers  | NOUN     | 0.420

Final Test Accuracy: 0.926
